# 유사한 단어 찾기 게임

1. 사전 학습된 모델 또는 적절한 데이터셋을 찾는다.
2. 워드 임베딩 모델을 학습시킨다.
3. 단어 유사도가 0.8 이상인 A, B를 랜덤 추출한다.
4. A, B와 대응되는 C를 추출한다.
5. D를 입력 받는다.

=>
A:B = C:D 관계에 대응하는 D를 찾는 게임을 만든다.
ex) A: 산, B: 바다, C: 나무, D: 물

**<출력 예시>**

관계 [ 수긍 : 추락 = 대사관 : ? ]<br>
모델이 예측한 가장 적합한 단어: 잠입<br>
당신의 답변과 모델 예측의 유사도: 0.34<br>
아쉽네요. 더 생각해보세요.

In [22]:
import pandas as pd

splits = {'train': 'dp/train-00000-of-00001.parquet', 'validation': 'dp/validation-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/klue/klue/" + splits["train"])

In [23]:
df = df['sentence']

In [24]:
from lxml import etree
import re
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

In [25]:
from konlpy.tag import Okt
from tqdm import tqdm
import re

okt = Okt()

# 기존 불용어 + 확장
ko_stopwords = [
    "은","는","이","가","을","를","과","와","들","도","부터","까지","에","나","너","그","걔","얘",
    "다","하다","되다","같다","있다",
    # 의미 없는 명사성 단어 추가
    "대해","위해","통해","정도","부분","경우","사실","때문","이번","이번에","관련"
]

preprocessed_data = []

for sentence in tqdm(df):
    sentence = re.sub(r"[a-zA-Z]", " ", sentence)   # 영문 제거
    sentence = re.sub(r"\d+", " ", sentence)        # 숫자 제거
    sentence = re.sub(r"[^가-힣\s]", " ", sentence) # 특수문자 제거

    # 품사 태깅
    morphs = okt.pos(sentence, stem=True)

    # 명사만 추출 + 불용어 제거 + 길이 2 이상
    tokens = [
        word for word, tag in morphs
        if tag in ["Noun", "ProperNoun"]
        and word not in ko_stopwords
        and len(word) > 1
    ]

    preprocessed_data.append(tokens)


100%|██████████| 10000/10000 [00:14<00:00, 673.55it/s]


In [26]:
from gensim.models import Word2Vec

model = Word2Vec(
    sentences=preprocessed_data, # corpus
    vector_size=10000,                # 임베딩 벡터 차원
    sg=0,                             # 학습 알고리즘 (0:CBOW, 1:Skip-gram)
    window=5,                         # 주변 단어 수 (앞뒤로 n개 사용) -> 이게 왜 5로 설정했는지 다시 확인
    min_count=5                       # 최소 빈도
)

In [27]:
import pandas as pd

pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key).head(10)

,0,1,2,3,4,5,6,7,8,9,...,9990,9991,9992,9993,9994,9995,9996,9997,9998,9999
숙소,0.006360,0.034836,0.002246,-0.022570,-0.016173,0.001676,0.037153,0.003430,0.007802,-0.009089,...,0.052575,-0.057233,0.002021,-0.003551,-0.021515,0.036365,0.016453,-0.046674,-0.007898,-0.012811
위치,0.005807,0.031270,0.002036,-0.020369,-0.014616,0.001450,0.033280,0.002912,0.007128,-0.008085,...,0.047407,-0.051629,0.001835,-0.003153,-0.019416,0.032677,0.014936,-0.042122,-0.007111,-0.011610
호스트,0.005362,0.029771,0.001865,-0.019527,-0.013791,0.001431,0.031789,0.002896,0.006634,-0.007743,...,0.044689,-0.048922,0.001877,-0.002927,-0.018212,0.031149,0.014011,-0.039950,-0.006884,-0.011042
지난,0.011002,0.060218,0.003546,-0.039038,-0.027939,0.003433,0.064183,0.006000,0.013347,-0.015445,...,0.090169,-0.098614,0.003911,-0.005885,-0.036721,0.062337,0.028317,-0.080019,-0.013983,-0.021929
정말,0.004612,0.025772,0.001719,-0.016818,-0.011806,0.001292,0.027470,0.002420,0.005717,-0.006750,...,0.038867,-0.042315,0.001430,-0.002695,-0.015887,0.026841,0.012241,-0.034658,-0.005879,-0.009498
시간,0.008936,0.048432,0.002863,-0.031386,-0.022569,0.002594,0.051481,0.004835,0.010742,-0.012311,...,0.072589,-0.079552,0.003048,-0.004845,-0.029674,0.050323,0.022638,-0.064542,-0.011366,-0.017614
사진,0.006547,0.035790,0.002048,-0.023140,-0.016648,0.001929,0.038069,0.003424,0.007925,-0.009196,...,0.053731,-0.058673,0.002324,-0.003524,-0.021985,0.037010,0.016940,-0.047718,-0.008185,-0.013004
여행,0.005956,0.032014,0.001903,-0.020862,-0.014765,0.001837,0.033979,0.003022,0.007198,-0.008203,...,0.048022,-0.052318,0.001876,-0.003202,-0.019580,0.033324,0.014955,-0.042761,-0.007399,-0.011838
매우,0.005171,0.028054,0.001669,-0.018323,-0.012942,0.001355,0.029816,0.002626,0.006418,-0.007285,...,0.042316,-0.046301,0.001551,-0.002839,-0.017385,0.029253,0.013216,-0.037604,-0.006433,-0.010295
한국,0.010315,0.056064,0.003260,-0.036394,-0.025975,0.003177,0.059828,0.005646,0.012537,-0.014256,...,0.083954,-0.092103,0.003594,-0.005509,-0.034299,0.058228,0.026358,-0.074761,-0.013160,-0.020297


In [28]:
# model : Word2Vec
model.wv.most_similar('남자')

[('미국', 0.9999877214431763),
 ('대한', 0.9999877214431763),
 ('이후', 0.9999874830245972),
 ('한국', 0.9999874234199524),
 ('지난', 0.9999874234199524),
 ('정부', 0.9999873042106628),
 ('지난해', 0.9999871850013733),
 ('내용', 0.9999871850013733),
 ('당시', 0.9999871850013733),
 ('문제', 0.9999871253967285)]

In [31]:
import random

kv = model.wv

def play():
    vocab = list(kv.key_to_index.keys())

    # A, B 두 단어 랜덤 선택한다.
    A, B = random.sample(vocab, 2)

    # A와 가장 유사한 단어 C 선택한다.
    try:
        C = kv.most_similar(A, topn=1)[0][0]
    except KeyError:
        print("해당 단어로는 유사도 계산 불가. 다시 실행하세요.")
        return

    # 모델 예측: A:B = C:?
    try:
        pred = kv.most_similar(positive=[B, C], negative=[A], topn=1)[0][0]
    except KeyError:
        print("관계 계산 불가. 다시 실행하세요.")
        return

    print(f"관계 [ {A} : {B} = {C} : ? ]")
    print(f"모델이 예측한 가장 적합한 단어: {pred}")

    user = input("D를 입력하세요: ").strip()
    print(f"당신이 선택한 단어: {user}")

    if user in kv and pred in kv:
        sim = kv.similarity(user, pred)
        print(f"당신의 답변과 모델 예측의 유사도: {sim:.2f}")
        if sim < 0.7:
            print("아쉽네요. 더 생각해보세요.")
    else:
        print("사전에 없는 단어라 유사도 계산 불가")

In [32]:
play()

관계 [ 배달 : 요청 = 대한 : ? ]
모델이 예측한 가장 적합한 단어: 문제
당신이 선택한 단어: 과자
당신의 답변과 모델 예측의 유사도: 1.00
